# Vesuvius V11 - Topology-Preserving Training

## Implementation Checklist

| Component | Setting | Status |
|-----------|---------|--------|
| **Loss Stack** | Dice(0.25) + BCE(0.1) + clDice(0.3) + Surface(0.15) + Topo(0.2) | ✅ |
| **Loss Schedule** | ALL losses from epoch 0 (no progressive) | ✅ |
| **Augmentation** | Mild affine, elastic(σ≤2), flips, occlusion | ✅ |
| **Architecture** | U-Net + residual + attention + hybrid convs + GroupNorm | ✅ |
| **Surface Block** | SurfaceRefinementBlock in decoder | ✅ |
| **Optimizer** | AdamW(lr=3e-4, wd=1e-2) + warmup + cosine | ✅ |
| **Precision** | bfloat16 (no scaler needed) | ✅ |
| **Grad Clip** | 1.0 | ✅ |

---

In [1]:
!pip install imagecodecs -q

   ╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.3/24.7 MB 10.3 MB/s eta 0:00:03

   ━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/24.7 MB 58.3 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━ 17.5/24.7 MB 188.8 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━ 23.2/24.7 MB 186.6 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.7/24.7 MB 88.2 MB/s eta 0:00:00


In [2]:
# =============================================================================
# CELL 1: IMPORTS & CONFIG
# =============================================================================

import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import gc
import json
import random
import warnings
from pathlib import Path
from typing import List, Tuple, Dict, Optional
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from PIL import Image, ImageSequence
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast

from scipy import ndimage
from scipy.ndimage import distance_transform_edt
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings('ignore')

@dataclass
class Config:
    # Data paths
    DATA_ROOT: Path = Path("/kaggle/input/datasets/asharamkanderiwal/aimo-dataset")
    CHECKPOINT_DIR: Path = Path("/kaggle/working/checkpoints_v11")  # Save checkpoints here
    LOAD_CHECKPOINT: Path = Path("/kaggle/input/models/asharamkanderiwal/v11-v2/pytorch/default/5/checkpoints_v11")  # Set to checkpoint path to resume training
    
    # ==========================================================================
    # MODEL CONFIG - 6 stage, 192 patch
    # ==========================================================================
    PATCH_SIZE: Tuple[int, int, int] = (192, 192, 192)
    FEATURES: List[int] = field(default_factory=lambda: [32, 64, 128, 256, 320, 320])
    N_BLOCKS: List[int] = field(default_factory=lambda: [1, 2, 3, 4, 6, 6])
    USE_ATTENTION: bool = True
    USE_HYBRID_CONV: bool = True
    USE_SURFACE_REFINEMENT: bool = True
    USE_DEEP_SUPERVISION: bool = True
    
    # ==========================================================================
    # TRAINING CONFIG
    # ==========================================================================
    EPOCHS_PER_FOLD: int = 800
    BATCH_SIZE: int = 4
    NUM_WORKERS: int = 16
    PREFETCH_FACTOR: int = 4
    
    # ==========================================================================
    # OPTIMIZER (from report)
    # ==========================================================================
    LEARNING_RATE: float = 3e-4
    WEIGHT_DECAY: float = 1e-2
    WARMUP_EPOCHS: int = 5
    ETA_MIN: float = 1e-6
    GRADIENT_CLIP: float = 1.0
    
    # ==========================================================================
    # LOSS WEIGHTS (ALL from epoch 0)
    # ==========================================================================
    DICE_WEIGHT: float = 0.25
    BCE_WEIGHT: float = 0.10
    CLDICE_WEIGHT: float = 0.30
    SURFACE_WEIGHT: float = 0.15
    TOPO_WEIGHT: float = 0.20
    
    # ==========================================================================
    # AUGMENTATION (mild, surface-preserving)
    # ==========================================================================
    AUG_FLIP: bool = True
    AUG_ROTATE: bool = True
    AUG_ELASTIC: bool = True
    AUG_ELASTIC_SIGMA: float = 2.0
    AUG_AFFINE: bool = True
    AUG_AFFINE_SCALE: Tuple[float, float] = (0.9, 1.1)
    AUG_NOISE: bool = True
    AUG_CONTRAST: bool = True
    AUG_BRIGHTNESS: bool = True
    AUG_OCCLUSION: bool = True
    
    # ==========================================================================
    # VALIDATION & CV
    # ==========================================================================
    VAL_EVERY: int = 5
    SAVE_EVERY: int = 10
    N_FOLDS: int = 3
    CV_SEED: int = 42
    
    # ==========================================================================
    # DATA
    # ==========================================================================
    PATCHES_PER_VOLUME: int = 12
    FG_OVERSAMPLE_RATIO: float = 0.6
    
    # Device & precision
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"
    USE_BFLOAT16: bool = True
    SEED: int = 42
    
    def __post_init__(self):
        self.CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

cfg = Config()

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

set_seed(cfg.SEED)

print("="*70)
print("V11 - TOPOLOGY-PRESERVING TRAINING")
print("="*70)
print(f"Patch: {cfg.PATCH_SIZE} | BS: {cfg.BATCH_SIZE} | Epochs: {cfg.EPOCHS_PER_FOLD}")
print(f"Stages: {len(cfg.FEATURES)} | Features: {cfg.FEATURES}")
print("="*70)
print("Loss weights (ALL from epoch 0):")
print(f"  Dice={cfg.DICE_WEIGHT}, BCE={cfg.BCE_WEIGHT}, clDice={cfg.CLDICE_WEIGHT}")
print(f"  Surface={cfg.SURFACE_WEIGHT}, Topo={cfg.TOPO_WEIGHT}")
print("="*70)
print(f"Checkpoint save dir: {cfg.CHECKPOINT_DIR}")
print(f"Load checkpoint: {cfg.LOAD_CHECKPOINT}")
print("="*70)
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    if torch.cuda.is_bf16_supported():
        print("bfloat16: SUPPORTED")
    else:
        print("bfloat16: NOT SUPPORTED - using float16")
        cfg.USE_BFLOAT16 = False
print("="*70)

V11 - TOPOLOGY-PRESERVING TRAINING
Patch: (192, 192, 192) | BS: 4 | Epochs: 800
Stages: 6 | Features: [32, 64, 128, 256, 320, 320]
Loss weights (ALL from epoch 0):
  Dice=0.25, BCE=0.1, clDice=0.3
  Surface=0.15, Topo=0.2
Checkpoint save dir: /kaggle/working/checkpoints_v11
Load checkpoint: /kaggle/input/models/asharamkanderiwal/v11-v2/pytorch/default/5/checkpoints_v11
GPU: NVIDIA H100 80GB HBM3
bfloat16: SUPPORTED


In [3]:
# =============================================================================
# CELL 2: STRATIFIED VOLUME FOLDS
# =============================================================================

def make_stratified_volume_folds(
    csv_path: Path,
    images_dir: Path,
    labels_dir: Path,
    n_splits: int = 3,
    seed: int = 42
) -> List[Tuple[List[str], List[str]]]:
    """Create stratified volume-level folds using scroll_id."""
    df = pd.read_csv(csv_path)
    
    valid_mask = df['id'].apply(
        lambda x: (images_dir / f"{x}.tif").exists() and (labels_dir / f"{x}.tif").exists()
    )
    df = df[valid_mask].reset_index(drop=True)
    
    print(f"Found {len(df)} valid volumes")
    
    if 'scroll_id' in df.columns:
        strat_col = df['scroll_id'].values
    else:
        strat_col = df['id'].apply(lambda x: x.split('_')[0] if '_' in x else 'unknown').values
    
    print(f"Scroll distribution: {pd.Series(strat_col).value_counts().to_dict()}")
    
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    
    splits = []
    for fold, (train_idx, val_idx) in enumerate(skf.split(df['id'], strat_col)):
        train_ids = df.iloc[train_idx]['id'].tolist()
        val_ids = df.iloc[val_idx]['id'].tolist()
        assert len(set(train_ids) & set(val_ids)) == 0, "Train/val overlap!"
        splits.append((train_ids, val_ids))
        print(f"Fold {fold}: Train={len(train_ids)}, Val={len(val_ids)}")
    
    return splits

# Create folds
train_csv = cfg.DATA_ROOT / "train.csv"
train_images = cfg.DATA_ROOT / "train_images"
train_labels = cfg.DATA_ROOT / "train_labels"

if train_csv.exists():
    FOLD_SPLITS = make_stratified_volume_folds(
        train_csv, train_images, train_labels,
        n_splits=cfg.N_FOLDS, seed=cfg.CV_SEED
    )
else:
    print("train.csv not found - test mode")
    FOLD_SPLITS = []

Found 786 valid volumes
Scroll distribution: {34117: 376, 35360: 170, 26010: 129, 26002: 82, 44430: 16, 53997: 13}
Fold 0: Train=524, Val=262
Fold 1: Train=524, Val=262
Fold 2: Train=524, Val=262


In [4]:
# =============================================================================
# CELL 3: LOSS FUNCTIONS (ALL from epoch 0) — GPU-OPTIMIZED FOR H100
# =============================================================================

def soft_skeletonize(x, num_iter=5):
    """Differentiable soft skeletonization at half resolution - optimal for H100."""
    orig_shape = x.shape[2:]
    # Half resolution - good balance of quality vs speed on H100
    x = F.interpolate(x, scale_factor=0.5, mode='trilinear', align_corners=False)
    for _ in range(num_iter):
        min_pool = -F.max_pool3d(-x, 3, stride=1, padding=1)
        max_min_pool = F.max_pool3d(min_pool, 3, stride=1, padding=1)
        x = F.relu(x - max_min_pool)
    # Upscale back
    x = F.interpolate(x, size=orig_shape, mode='trilinear', align_corners=False)
    return x


class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-5):
        super().__init__()
        self.smooth = smooth
    
    def forward(self, pred, target, mask=None):
        pred = torch.sigmoid(pred)
        if mask is not None:
            pred = pred * (1 - mask)
            target = target * (1 - mask)
        inter = (pred * target).sum()
        union = pred.sum() + target.sum()
        return 1 - (2 * inter + self.smooth) / (union + self.smooth)


class clDiceLoss(nn.Module):
    """Centerline Dice — encourages continuity without bridges."""
    def __init__(self, num_iter=5, smooth=1e-5):
        super().__init__()
        self.num_iter = num_iter
        self.smooth = smooth
    
    def forward(self, pred, target, mask=None):
        pred_sig = torch.sigmoid(pred)
        if mask is not None:
            pred_sig = pred_sig * (1 - mask)
            target = target * (1 - mask)
        
        skel_pred = soft_skeletonize(pred_sig, self.num_iter)
        # Target skeleton doesn't need gradients
        with torch.no_grad():
            skel_target = soft_skeletonize(target, self.num_iter)
        
        tprec = ((skel_pred * target).sum() + self.smooth) / (skel_pred.sum() + self.smooth)
        tsens = ((skel_target * pred_sig).sum() + self.smooth) / (skel_target.sum() + self.smooth)
        
        cl_dice = 2 * tprec * tsens / (tprec + tsens + self.smooth)
        return 1 - cl_dice


def gpu_approx_signed_distance(target, num_iters=5):
    """
    GPU-native approximate signed distance map via iterative morphological dilation.
    5 iterations for better boundary precision on H100.
    """
    binary = (target > 0.5).float()
    inv_binary = 1.0 - binary
    
    # Approximate distance to foreground boundary (for background pixels)
    dist_bg = torch.zeros_like(binary)
    frontier = inv_binary.clone()
    for i in range(1, num_iters + 1):
        dilated = F.max_pool3d(binary, 3, stride=1, padding=1)
        new_frontier = (dilated > 0.5) & (frontier > 0.5)
        dist_bg = dist_bg + new_frontier.float()
        frontier = frontier * (1.0 - new_frontier.float())
        binary = dilated
    dist_bg = dist_bg + frontier * (num_iters + 1)
    
    # Approximate distance to background boundary (for foreground pixels)
    binary_fg = (target > 0.5).float()
    dist_fg = torch.zeros_like(binary_fg)
    frontier_fg = binary_fg.clone()
    inv_fg = 1.0 - binary_fg
    for i in range(1, num_iters + 1):
        dilated = F.max_pool3d(inv_fg, 3, stride=1, padding=1)
        new_frontier = (dilated > 0.5) & (frontier_fg > 0.5)
        dist_fg = dist_fg + new_frontier.float()
        frontier_fg = frontier_fg * (1.0 - new_frontier.float())
        inv_fg = dilated
    dist_fg = dist_fg + frontier_fg * (num_iters + 1)
    
    # Signed distance: positive outside, negative inside
    is_fg = (target > 0.5).float()
    signed_dist = dist_bg * (1.0 - is_fg) - dist_fg * is_fg
    max_val = signed_dist.abs().amax(dim=(1, 2, 3, 4), keepdim=True) + 1e-8
    signed_dist = signed_dist / max_val
    return signed_dist


class SurfaceLoss(nn.Module):
    """Surface Loss — GPU-native boundary alignment via approximate distance maps."""
    def forward(self, pred, target, mask=None):
        pred_sig = torch.sigmoid(pred)
        
        with torch.no_grad():
            dist_map = gpu_approx_signed_distance(target, num_iters=5)
        
        if mask is not None:
            pred_sig = pred_sig * (1 - mask)
            dist_map = dist_map * (1 - mask)
        
        return (pred_sig * dist_map).mean()


class TopoLoss(nn.Module):
    """Topology Loss — penalizes holes/bridges via Laplacian."""
    def __init__(self, sigma=2.0):
        super().__init__()
        self.sigma = sigma
        kernel = torch.tensor([
            [[0, 0, 0], [0, 1, 0], [0, 0, 0]],
            [[0, 1, 0], [1, -6, 1], [0, 1, 0]],
            [[0, 0, 0], [0, 1, 0], [0, 0, 0]]
        ], dtype=torch.float32).view(1, 1, 3, 3, 3)
        self.register_buffer('laplacian_kernel', kernel)
    
    def forward(self, pred, target, mask=None):
        pred_sig = torch.sigmoid(pred)
        
        if mask is not None:
            pred_sig = pred_sig * (1 - mask)
            target = target * (1 - mask)
        
        kernel = self.laplacian_kernel.to(dtype=pred.dtype, device=pred.device)
        lap_pred = F.conv3d(pred_sig, kernel, padding=1)
        lap_target = F.conv3d(target, kernel, padding=1)
        
        topo_diff = (lap_pred - lap_target).abs()
        weight = torch.exp(-self.sigma * target)
        
        return (topo_diff * weight).mean()


class CombinedLossV11(nn.Module):
    """V11 Combined Loss — ALL losses from epoch 0."""
    def __init__(self, dice_w=0.25, bce_w=0.10, cldice_w=0.30, surface_w=0.15, topo_w=0.20):
        super().__init__()
        self.dice_w = dice_w
        self.bce_w = bce_w
        self.cldice_w = cldice_w
        self.surface_w = surface_w
        self.topo_w = topo_w
        
        self.dice_loss = DiceLoss()
        self.cldice_loss = clDiceLoss()
        self.surface_loss = SurfaceLoss()
        self.topo_loss = TopoLoss()
        self.ds_weights = [0.5, 0.25, 0.125]
    
    def forward(self, output, target, ignore_mask, epoch=None):
        if isinstance(output, dict):
            pred = output['output']
            deep = output.get('deep', [])
        else:
            pred = output
            deep = []
        
        dice = self.dice_loss(pred, target, ignore_mask)
        bce = F.binary_cross_entropy_with_logits(
            pred * (1 - ignore_mask) if ignore_mask is not None else pred,
            target * (1 - ignore_mask) if ignore_mask is not None else target,
        )
        cldice = self.cldice_loss(pred, target, ignore_mask)
        surface = self.surface_loss(pred, target, ignore_mask)
        topo = self.topo_loss(pred, target, ignore_mask)
        
        total = (self.dice_w * dice + self.bce_w * bce + self.cldice_w * cldice +
                 self.surface_w * surface + self.topo_w * topo)
        
        for i, ds in enumerate(deep):
            if i >= len(self.ds_weights): break
            ds_target = F.interpolate(target, size=ds.shape[2:], mode='nearest')
            ds_mask = F.interpolate(ignore_mask, size=ds.shape[2:], mode='nearest') if ignore_mask is not None else None
            total = total + self.ds_weights[i] * self.dice_loss(ds, ds_target, ds_mask)
        
        return {'total': total, 'dice': dice, 'bce': bce,
                'cldice': cldice, 'surface': surface, 'topo': topo}

print("Loss Stack ready (ALL from epoch 0) — OPTIMIZED FOR H100")
print("H100 settings:")
print("  - soft_skeletonize: half resolution, 5 iterations (quality)")
print("  - gpu_approx_signed_distance: 5 iterations (better boundary precision)")

Loss Stack ready (ALL from epoch 0) — OPTIMIZED FOR H100
H100 settings:
  - soft_skeletonize: half resolution, 5 iterations (quality)
  - gpu_approx_signed_distance: 5 iterations (better boundary precision)


In [5]:
# =============================================================================
# CELL 4: MODEL ARCHITECTURE
# =============================================================================

def get_num_groups(channels, max_groups=32):
    for g in [max_groups, 16, 8, 4, 2, 1]:
        if channels % g == 0:
            return g
    return 1


class HybridConv3d(nn.Module):
    """3x3x1 + 1x1x3 for XY/Z balance."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        mid_ch = out_ch // 2
        self.conv_xy = nn.Conv3d(in_ch, mid_ch, kernel_size=(1, 3, 3), padding=(0, 1, 1), bias=False)
        self.conv_z = nn.Conv3d(in_ch, out_ch - mid_ch, kernel_size=(3, 1, 1), padding=(1, 0, 0), bias=False)
        self.norm = nn.GroupNorm(get_num_groups(out_ch), out_ch)
        self.act = nn.LeakyReLU(0.01, inplace=True)
    
    def forward(self, x):
        return self.act(self.norm(torch.cat([self.conv_xy(x), self.conv_z(x)], dim=1)))


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, use_hybrid=False):
        super().__init__()
        if use_hybrid:
            self.conv = HybridConv3d(in_ch, out_ch)
        else:
            self.conv = nn.Sequential(
                nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False),
                nn.GroupNorm(get_num_groups(out_ch), out_ch),
                nn.LeakyReLU(0.01, inplace=True),
            )
    
    def forward(self, x):
        return self.conv(x)


class MultiScaleResBlock(nn.Module):
    def __init__(self, channels, scales=2, use_hybrid=False):
        super().__init__()
        self.scales = scales
        self.width = channels // scales
        self.convs = nn.ModuleList([ConvBlock(self.width, self.width, use_hybrid=use_hybrid) for _ in range(scales - 1)])
        self.norm = nn.GroupNorm(get_num_groups(channels), channels)
    
    def forward(self, x):
        splits = torch.chunk(x, self.scales, dim=1)
        outputs = [splits[0]]
        for i, conv in enumerate(self.convs):
            out = conv(splits[i + 1] + outputs[-1]) if i > 0 else conv(splits[i + 1])
            outputs.append(out)
        return x + self.norm(torch.cat(outputs, dim=1))


class AttentionBlock(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.gap = nn.AdaptiveAvgPool3d(1)
        self.fc1 = nn.Linear(channels, max(channels // reduction, 8))
        self.fc2 = nn.Linear(max(channels // reduction, 8), channels)
        self.conv_spatial = nn.Conv3d(2, 1, kernel_size=7, padding=3, bias=False)
    
    def forward(self, x):
        b, c = x.shape[:2]
        ca = torch.sigmoid(self.fc2(F.relu(self.fc1(self.gap(x).view(b, c))))).view(b, c, 1, 1, 1)
        x_ca = x * ca
        sa = torch.sigmoid(self.conv_spatial(torch.cat([x_ca.mean(1, keepdim=True), x_ca.max(1, keepdim=True)[0]], 1)))
        return x_ca * sa


class SurfaceRefinementBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.edge_conv = nn.Conv3d(in_ch, in_ch, 3, padding=1, bias=False)
        self.refine = nn.Sequential(
            nn.Conv3d(in_ch * 2, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(get_num_groups(out_ch), out_ch),
            nn.LeakyReLU(0.01, inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(get_num_groups(out_ch), out_ch),
            nn.LeakyReLU(0.01, inplace=True),
        )
    
    def forward(self, x):
        return self.refine(torch.cat([x, torch.abs(self.edge_conv(x))], dim=1))


class TopoPreservingUNet3D(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, features=None, n_blocks=None,
                 use_attention=True, use_hybrid_conv=True, use_surface_refinement=True, use_deep_supervision=True):
        super().__init__()
        features = features or [32, 64, 128, 256, 320, 320]
        n_blocks = n_blocks or [1, 2, 3, 4, 6, 6]
        self.features = features
        self.use_deep_supervision = use_deep_supervision
        
        self.encoders = nn.ModuleList()
        self.attentions = nn.ModuleList()
        self.pools = nn.ModuleList()
        
        for i, (feat, nb) in enumerate(zip(features, n_blocks)):
            in_channels = in_ch if i == 0 else features[i-1]
            layers = [ConvBlock(in_channels, feat, use_hybrid=use_hybrid_conv and i > 0)]
            layers.extend([MultiScaleResBlock(feat, scales=2, use_hybrid=use_hybrid_conv) for _ in range(nb)])
            self.encoders.append(nn.Sequential(*layers))
            self.attentions.append(AttentionBlock(feat) if use_attention and i >= 2 else nn.Identity())
            if i < len(features) - 1:
                self.pools.append(nn.Conv3d(feat, feat, 2, stride=2))
        
        self.ups = nn.ModuleList()
        self.dec_convs = nn.ModuleList()
        
        for i in range(len(features)-2, -1, -1):
            self.ups.append(nn.ConvTranspose3d(features[i+1], features[i], 2, stride=2))
            if use_surface_refinement and i == 0:
                self.dec_convs.append(SurfaceRefinementBlock(features[i]*2, features[i]))
            else:
                self.dec_convs.append(nn.Sequential(
                    ConvBlock(features[i]*2, features[i], use_hybrid=use_hybrid_conv),
                    MultiScaleResBlock(features[i], scales=2, use_hybrid=use_hybrid_conv),
                ))
        
        self.final = nn.Conv3d(features[0], out_ch, 1)
        if use_deep_supervision:
            self.ds_heads = nn.ModuleList([nn.Conv3d(features[-(i+2)], out_ch, 1) for i in range(min(3, len(features)-1))])
    
    def forward(self, x):
        enc_features = []
        for i, (enc, att) in enumerate(zip(self.encoders, self.attentions)):
            x = att(enc(x))
            enc_features.append(x)
            if i < len(self.pools): x = self.pools[i](x)
        
        enc_features = enc_features[::-1]
        x = enc_features[0]
        ds_outputs = []
        
        for i, (up, dec) in enumerate(zip(self.ups, self.dec_convs)):
            x = up(x)
            skip = enc_features[i+1]
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
            x = dec(torch.cat([x, skip], dim=1))
            if self.use_deep_supervision and self.training and i < len(self.ds_heads):
                ds_outputs.append(self.ds_heads[i](x))
        
        out = self.final(x)
        return {'output': out, 'deep': ds_outputs} if self.use_deep_supervision and self.training else out


def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

_m = TopoPreservingUNet3D(features=cfg.FEATURES, n_blocks=cfg.N_BLOCKS)
print(f"TopoPreservingUNet3D: {count_params(_m)/1e6:.1f}M params")
print(f"  Stages: {len(cfg.FEATURES)} | Features: {cfg.FEATURES}")
del _m

TopoPreservingUNet3D: 10.0M params
  Stages: 6 | Features: [32, 64, 128, 256, 320, 320]


In [6]:
# =============================================================================
# CELL 5: GPU AUGMENTATIONS (all ops on GPU, no CPU bottleneck)
# =============================================================================

class GPUAugmentation3D(nn.Module):
    """
    GPU-native 3D augmentation using pure PyTorch ops.
    Replaces scipy-based CPU augmentations for ~10-50x speedup.
    """
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg

    @torch.no_grad()
    def forward(self, img, lbl, ignore):
        """
        Args:
            img: (B, 1, D, H, W) float tensor on GPU
            lbl: (B, 1, D, H, W) float tensor on GPU
            ignore: (B, 1, D, H, W) float tensor on GPU
        Returns:
            Augmented (img, lbl, ignore) tensors
        """
        B = img.shape[0]
        device = img.device
        dtype = img.dtype  # Preserve input dtype (bfloat16)

        # --- Spatial augmentations (applied to all channels equally) ---

        # Flip (per-sample random)
        if self.cfg.AUG_FLIP:
            for ax in [2, 3, 4]:  # D, H, W
                mask = torch.rand(B, device=device) > 0.5
                if mask.any():
                    idx = mask.nonzero(as_tuple=True)[0]
                    img[idx] = torch.flip(img[idx], [ax])
                    lbl[idx] = torch.flip(lbl[idx], [ax])
                    ignore[idx] = torch.flip(ignore[idx], [ax])

        # Rotate 90 in HW plane (per-sample random)
        if self.cfg.AUG_ROTATE:
            for b in range(B):
                k = torch.randint(0, 4, (1,)).item()
                if k > 0:
                    img[b] = torch.rot90(img[b], k, [2, 3])     # H, W dims
                    lbl[b] = torch.rot90(lbl[b], k, [2, 3])
                    ignore[b] = torch.rot90(ignore[b], k, [2, 3])

        # Elastic + Affine via grid_sample (batched, very fast on GPU)
        if self.cfg.AUG_ELASTIC or self.cfg.AUG_AFFINE:
            img, lbl, ignore = self._elastic_affine(img, lbl, ignore)

        # --- Intensity augmentations (image only) ---

        # Gaussian noise
        if self.cfg.AUG_NOISE:
            noise_mask = torch.rand(B, device=device) > 0.5
            if noise_mask.any():
                idx = noise_mask.nonzero(as_tuple=True)[0]
                img[idx] = img[idx] + torch.randn_like(img[idx]) * 0.05

        # Contrast
        if self.cfg.AUG_CONTRAST:
            contrast_mask = torch.rand(B, device=device) > 0.5
            if contrast_mask.any():
                idx = contrast_mask.nonzero(as_tuple=True)[0]
                mean = img[idx].mean(dim=(1, 2, 3, 4), keepdim=True)
                scale = torch.empty(len(idx), 1, 1, 1, 1, device=device, dtype=dtype).uniform_(0.8, 1.2)
                img[idx] = (img[idx] - mean) * scale + mean

        # Brightness
        if self.cfg.AUG_BRIGHTNESS:
            bright_mask = torch.rand(B, device=device) > 0.5
            if bright_mask.any():
                idx = bright_mask.nonzero(as_tuple=True)[0]
                shift = torch.empty(len(idx), 1, 1, 1, 1, device=device, dtype=dtype).uniform_(-0.1, 0.1)
                img[idx] = img[idx] + shift

        # Occlusion (random cubic patches zeroed out)
        if self.cfg.AUG_OCCLUSION:
            occ_mask = torch.rand(B, device=device) > 0.7
            if occ_mask.any():
                idx = occ_mask.nonzero(as_tuple=True)[0]
                img = self._apply_occlusion(img, idx)

        return img, lbl, ignore

    def _elastic_affine(self, img, lbl, ignore):
        """Combined elastic + affine deformation using F.grid_sample."""
        B, _, D, H, W = img.shape
        device = img.device
        dtype = img.dtype  # Preserve input dtype (bfloat16)

        # Create base grid: (B, D, H, W, 3) in [-1, 1] - MUST match input dtype
        grid = torch.stack(torch.meshgrid(
            torch.linspace(-1, 1, D, device=device, dtype=dtype),
            torch.linspace(-1, 1, H, device=device, dtype=dtype),
            torch.linspace(-1, 1, W, device=device, dtype=dtype),
            indexing='ij'
        ), dim=-1).unsqueeze(0).expand(B, -1, -1, -1, -1).clone()

        # Elastic deformation
        if self.cfg.AUG_ELASTIC:
            elastic_mask = torch.rand(B, device=device) > 0.5
            if elastic_mask.any():
                idx = elastic_mask.nonzero(as_tuple=True)[0]
                n_idx = len(idx)
                # Generate smooth random displacement at low res then upsample
                low_d, low_h, low_w = max(4, D // 16), max(4, H // 16), max(4, W // 16)
                sigma = self.cfg.AUG_ELASTIC_SIGMA
                # Scale displacement: sigma controls magnitude in normalized [-1,1] space
                disp_scale = sigma * 2.0 / max(D, H, W)
                noise = torch.randn(n_idx, 3, low_d, low_h, low_w, device=device, dtype=dtype) * disp_scale
                # Upsample to full resolution (smooth interpolation = smooth deformation)
                disp = F.interpolate(noise, size=(D, H, W), mode='trilinear', align_corners=False)
                # disp: (n_idx, 3, D, H, W) -> (n_idx, D, H, W, 3)
                disp = disp.permute(0, 2, 3, 4, 1)
                grid[idx] = grid[idx] + disp

        # Affine (scale)
        if self.cfg.AUG_AFFINE:
            affine_mask = torch.rand(B, device=device) > 0.5
            if affine_mask.any():
                idx = affine_mask.nonzero(as_tuple=True)[0]
                lo, hi = self.cfg.AUG_AFFINE_SCALE
                scales = torch.empty(len(idx), 1, 1, 1, 3, device=device, dtype=dtype).uniform_(lo, hi)
                grid[idx] = grid[idx] * scales

        # Clamp grid to valid range
        grid = grid.clamp(-1, 1)

        # Apply deformation - use grid_sample (very fast on GPU)
        # grid_sample expects grid in (B, D, H, W, 3) with order (x, y, z) but we have (z, y, x)
        # PyTorch grid_sample 3D: grid[..., 0] = W dim, grid[..., 1] = H dim, grid[..., 2] = D dim
        grid_sample = grid.flip(-1)  # (z,y,x) -> (x,y,z) for grid_sample

        img = F.grid_sample(img, grid_sample, mode='bilinear', padding_mode='reflection', align_corners=False)
        # Use nearest for labels to preserve discrete values
        lbl = F.grid_sample(lbl, grid_sample, mode='nearest', padding_mode='zeros', align_corners=False)
        ignore = F.grid_sample(ignore, grid_sample, mode='nearest', padding_mode='zeros', align_corners=False)

        return img, lbl, ignore

    def _apply_occlusion(self, img, idx):
        """Apply random cubic occlusion patches."""
        _, _, D, H, W = img.shape
        for b_idx in idx:
            for _ in range(3):
                if torch.rand(1).item() > 0.5:
                    size = torch.randint(5, 15, (1,)).item()
                    z = torch.randint(0, max(1, D - size), (1,)).item()
                    y = torch.randint(0, max(1, H - size), (1,)).item()
                    x = torch.randint(0, max(1, W - size), (1,)).item()
                    patch = img[b_idx, :, z:z+size, y:y+size, x:x+size]
                    img[b_idx, :, z:z+size, y:y+size, x:x+size] = patch.mean()
        return img

print("GPUAugmentation3D ready (all ops on GPU, bfloat16 compatible)")

GPUAugmentation3D ready (all ops on GPU, bfloat16 compatible)


In [7]:
# =============================================================================
# CELL 6: DATASET WITH ROBUST NORMALIZATION
# =============================================================================

def load_volume(path):
    try:
        import tifffile
        return tifffile.imread(str(path))
    except:
        im = Image.open(path)
        return np.stack([np.array(p) for p in ImageSequence.Iterator(im)], axis=0)


def robust_zscore_normalize(img, lower_percentile=0.5, upper_percentile=99.5):
    """
    Robust Z-score normalization with percentile clipping.
    
    This is the standard in medical imaging (nnU-Net, MONAI):
    1. Clip outliers using percentiles (handles artifacts, noise)
    2. Z-score normalize using clipped statistics
    
    Why this works better:
    - CT/X-ray data often has outliers (bright artifacts, air pockets)
    - Percentile clipping removes these before computing mean/std
    - Results in more stable training and better generalization
    """
    # Compute percentiles for clipping
    p_low = np.percentile(img, lower_percentile)
    p_high = np.percentile(img, upper_percentile)
    
    # Clip to remove outliers
    img_clipped = np.clip(img, p_low, p_high)
    
    # Compute statistics on clipped data (more robust)
    mean = img_clipped.mean()
    std = img_clipped.std()
    
    # Normalize (use clipped image)
    img_norm = (img_clipped - mean) / (std + 1e-8)
    
    return img_norm.astype(np.float32)


VOLUME_CACHE = {}
FG_COORDS_CACHE = {}

def preload_volumes(volume_ids, images_dir, labels_dir):
    """Preload volumes with robust normalization."""
    global VOLUME_CACHE, FG_COORDS_CACHE
    to_load = [vid for vid in volume_ids if vid not in VOLUME_CACHE]
    if not to_load:
        print(f"All {len(volume_ids)} volumes cached")
        return
    print(f"Preloading {len(to_load)} volumes with robust Z-score normalization...")
    for vid in tqdm(to_load, desc="Loading"):
        img = load_volume(Path(images_dir) / f"{vid}.tif").astype(np.float32)
        lbl = load_volume(Path(labels_dir) / f"{vid}.tif").astype(np.uint8)
        
        # Robust normalization (percentile clipping + Z-score)
        img = robust_zscore_normalize(img, lower_percentile=0.5, upper_percentile=99.5)
        
        VOLUME_CACHE[vid] = (img, lbl)
        fg = np.argwhere(lbl == 1)
        FG_COORDS_CACHE[vid] = fg[np.random.choice(len(fg), min(10000, len(fg)), replace=False)] if len(fg) > 0 else None
    print(f"Cached: {len(VOLUME_CACHE)} volumes ({sum(v[0].nbytes+v[1].nbytes for v in VOLUME_CACHE.values())/1e9:.1f} GB)")


class VesuviusDatasetV11(Dataset):
    def __init__(self, volume_ids, images_dir, labels_dir, patch_size=(192,192,192),
                 is_train=True, patches_per_volume=12, fg_oversample=0.6):
        self.patch_size = patch_size
        self.is_train = is_train
        self.patches_per_volume = patches_per_volume
        self.fg_oversample = fg_oversample
        self.volume_ids = volume_ids
        preload_volumes(volume_ids, images_dir, labels_dir)
        print(f"Dataset: {len(self)} samples ({'train' if is_train else 'val'})")
    
    def __len__(self):
        return len(self.volume_ids) * self.patches_per_volume
    
    def __getitem__(self, idx):
        vid = self.volume_ids[idx // self.patches_per_volume]
        img, lbl = VOLUME_CACHE[vid]
        d, h, w = img.shape
        pd, ph, pw = self.patch_size
        
        if d < pd or h < ph or w < pw:
            img = np.pad(img, ((0, max(0,pd-d)), (0, max(0,ph-h)), (0, max(0,pw-w))), mode='reflect')
            lbl = np.pad(lbl, ((0, max(0,pd-d)), (0, max(0,ph-h)), (0, max(0,pw-w))), mode='constant', constant_values=2)
            d, h, w = img.shape
        
        fg = FG_COORDS_CACHE.get(vid)
        if self.is_train and random.random() < self.fg_oversample and fg is not None and len(fg) > 0:
            c = fg[random.randint(0, len(fg)-1)]
            z, y, x = [max(0, min(c[i] - self.patch_size[i]//2, s - self.patch_size[i])) for i, s in enumerate([d,h,w])]
        else:
            z, y, x = [random.randint(0, max(0, s - p)) for s, p in zip([d,h,w], self.patch_size)]
        
        img_p = img[z:z+pd, y:y+ph, x:x+pw].copy()
        lbl_p = lbl[z:z+pd, y:y+ph, x:x+pw].copy()
        
        return {
            'image': torch.from_numpy(img_p).unsqueeze(0).float(),
            'label': torch.from_numpy((lbl_p == 1).astype(np.float32)).unsqueeze(0),
            'ignore_mask': torch.from_numpy((lbl_p == 2).astype(np.float32)).unsqueeze(0),
        }

print("VesuviusDatasetV11 ready with ROBUST NORMALIZATION")
print("Normalization: Percentile clipping (0.5-99.5%) + Z-score")
print("  - Removes outliers/artifacts before normalization")
print("  - More stable training, better generalization")

VesuviusDatasetV11 ready with ROBUST NORMALIZATION
Normalization: Percentile clipping (0.5-99.5%) + Z-score
  - Removes outliers/artifacts before normalization
  - More stable training, better generalization


In [8]:
# =============================================================================
# CELL 7: CHECKPOINT SAVE & LOAD (FIXED)
# =============================================================================

def save_checkpoint(path, model, optimizer, scheduler, epoch, best_dice, history, cfg):
    """
    Save training checkpoint with all state needed to resume.
    
    Saves:
    - Model weights
    - Optimizer state (momentum, etc.)
    - Scheduler state (current LR position)
    - Current epoch
    - Best dice score
    - Training history
    - Config for reproducibility
    """
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
        'best_dice': best_dice,
        'history': history,
        'config': {
            'features': cfg.FEATURES,
            'n_blocks': cfg.N_BLOCKS,
            'patch_size': cfg.PATCH_SIZE,
            'batch_size': cfg.BATCH_SIZE,
            'learning_rate': cfg.LEARNING_RATE,
        }
    }
    torch.save(checkpoint, path)
    print(f"Checkpoint saved: {path}")


def load_checkpoint(path, model, optimizer=None, scheduler=None, device='cuda'):
    """
    Load training checkpoint to resume training.
    
    Returns:
        start_epoch: Epoch to resume from (checkpoint epoch + 1)
        best_dice: Best dice score so far
        history: Training history list
    """
    print(f"Loading checkpoint: {path}")
    checkpoint = torch.load(path, map_location=device, weights_only=False)
    
    # Load model weights (handle torch.compile prefix)
    state_dict = checkpoint['model_state_dict']
    state_dict = {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()}
    model.load_state_dict(state_dict, strict=False)
    
    # Load optimizer state
    if optimizer and 'optimizer_state_dict' in checkpoint:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        print(f"  Optimizer state loaded")
    
    # Load scheduler state
    if scheduler and checkpoint.get('scheduler_state_dict'):
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        print(f"  Scheduler state loaded")
    
    start_epoch = checkpoint.get('epoch', 0) + 1
    best_dice = checkpoint.get('best_dice', 0)
    history = checkpoint.get('history', [])
    
    print(f"  Resuming from epoch {start_epoch}")
    print(f"  Best dice so far: {best_dice:.4f}")
    print(f"  History entries: {len(history)}")
    
    # Show config diff if available
    if 'config' in checkpoint:
        saved_cfg = checkpoint['config']
        print(f"  Saved config: patch={saved_cfg.get('patch_size')}, bs={saved_cfg.get('batch_size')}")
    
    return start_epoch, best_dice, history


def get_latest_checkpoint(checkpoint_dir, fold=None, load_dir=None):
    """
    Find the latest checkpoint to resume from.
    
    Priority:
    1. last_checkpoint.pth (always saved after each epoch)
    2. checkpoint_epoch_*.pth (periodic saves) - highest epoch number
    3. best_model.pth (fallback if no other checkpoints)
    
    Args:
        checkpoint_dir: Directory to save checkpoints (working dir)
        fold: If provided, look in fold_{fold}/ subdirectory
        load_dir: Optional separate directory to load from (e.g., Kaggle input dataset)
    
    Returns:
        Path to latest checkpoint or None
    """
    # Build list of directories to search (save dir first, then load dir)
    search_dirs = []
    
    save_dir = Path(checkpoint_dir)
    if fold is not None:
        save_dir = save_dir / f"fold_{fold}"
    search_dirs.append(save_dir)
    
    if load_dir:
        load_path = Path(load_dir)
        if fold is not None:
            load_path = load_path / f"fold_{fold}"
        search_dirs.append(load_path)
    
    for search_dir in search_dirs:
        if not search_dir.exists():
            continue
        
        # Priority 1: last_checkpoint.pth (most recent state)
        last_ckpt = search_dir / "last_checkpoint.pth"
        if last_ckpt.exists():
            print(f"Found last_checkpoint.pth in {search_dir}")
            return last_ckpt
        
        # Priority 2: checkpoint_epoch_*.pth (periodic saves) - find highest epoch
        epoch_ckpts = list(search_dir.glob("checkpoint_epoch_*.pth"))
        if epoch_ckpts:
            def get_epoch(p):
                try:
                    return int(p.stem.split('_')[-1])
                except:
                    return 0
            epoch_ckpts.sort(key=get_epoch, reverse=True)
            print(f"Found checkpoint_epoch_{get_epoch(epoch_ckpts[0])}.pth in {search_dir}")
            return epoch_ckpts[0]
        
        # Priority 3: best_model.pth (fallback)
        best_ckpt = search_dir / "best_model.pth"
        if best_ckpt.exists():
            print(f"Found best_model.pth in {search_dir} (no last/periodic checkpoint)")
            return best_ckpt
    
    print(f"No checkpoint found for fold {fold}")
    return None


print("Checkpoint utilities ready (FIXED)")
print("Checkpoint loading priority:")
print("  1. last_checkpoint.pth (saved every epoch)")
print("  2. checkpoint_epoch_*.pth (highest epoch)")
print("  3. best_model.pth (fallback)")
print(f"Save dir: {cfg.CHECKPOINT_DIR}")
print(f"Load from: {cfg.LOAD_CHECKPOINT}")

Checkpoint utilities ready (FIXED)
Checkpoint loading priority:
  1. last_checkpoint.pth (saved every epoch)
  2. checkpoint_epoch_*.pth (highest epoch)
  3. best_model.pth (fallback)
Save dir: /kaggle/working/checkpoints_v11
Load from: /kaggle/input/models/asharamkanderiwal/v11-v2/pytorch/default/5/checkpoints_v11


In [9]:
# =============================================================================
# CELL 8: VALIDATION
# =============================================================================

def compute_dice(pred, gt):
    inter = (pred & gt).sum()
    union = pred.sum() + gt.sum()
    return (2 * inter + 1e-5) / (union + 1e-5)


@torch.no_grad()
def validate_fast(model, loader, device, use_bf16=True):
    """Fast patch-based validation (Dice only)."""
    model.eval()
    total_dice = 0
    n = 0
    dtype = torch.bfloat16 if use_bf16 else torch.float32
    
    for batch in loader:
        images = batch['image'].to(device, dtype=dtype)
        labels = batch['label'].numpy()
        ignore = batch['ignore_mask'].numpy()
        
        with autocast(device_type='cuda', dtype=dtype):
            out = model(images)
            if isinstance(out, dict): out = out['output']
            probs = torch.sigmoid(out).float().cpu().numpy()
        
        for b in range(images.shape[0]):
            pred = (probs[b,0] > 0.5).astype(np.uint8)
            tgt = labels[b,0].astype(np.uint8)
            ign = ignore[b,0] > 0.5
            pred[ign] = 0
            tgt[ign] = 0
            total_dice += compute_dice(pred, tgt)
            n += 1
    
    return total_dice / max(n, 1)

print("Validation ready")

Validation ready


In [10]:
# =============================================================================
# CELL 9: TRAINING LOOP (FIXED - saves last checkpoint every epoch)
# =============================================================================

import sys
import time

def get_warmup_lr(epoch, warmup_epochs, base_lr):
    if epoch < warmup_epochs:
        return base_lr * (epoch + 1) / warmup_epochs
    return base_lr


def train_fold_v11(fold: int, train_ids: List[str], val_ids: List[str], resume_from: Path = None) -> Dict:
    """
    Train fold with checkpoint save/load support.
    
    Checkpoint strategy:
    - last_checkpoint.pth: Saved EVERY epoch (for exact resume)
    - best_model.pth: Saved when validation improves
    - checkpoint_epoch_N.pth: Saved every SAVE_EVERY epochs (backup)
    
    Args:
        fold: Fold number
        train_ids: Training volume IDs
        val_ids: Validation volume IDs
        resume_from: Path to checkpoint to resume from (optional)
    """
    print("="*70)
    print(f"FOLD {fold} TRAINING")
    print(f"Train: {len(train_ids)} | Val: {len(val_ids)}")
    print("="*70)

    train_images = cfg.DATA_ROOT / "train_images"
    train_labels = cfg.DATA_ROOT / "train_labels"

    # Datasets
    train_ds = VesuviusDatasetV11(train_ids, train_images, train_labels,
                                   patch_size=cfg.PATCH_SIZE, is_train=True,
                                   patches_per_volume=cfg.PATCHES_PER_VOLUME)
    val_ds = VesuviusDatasetV11(val_ids, train_images, train_labels,
                                 patch_size=cfg.PATCH_SIZE, is_train=False, patches_per_volume=4)

    train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True,
                              num_workers=cfg.NUM_WORKERS, pin_memory=True, drop_last=True,
                              persistent_workers=True, prefetch_factor=cfg.PREFETCH_FACTOR)
    val_loader = DataLoader(val_ds, batch_size=cfg.BATCH_SIZE, shuffle=False,
                            num_workers=4, pin_memory=True)

    # Model
    model = TopoPreservingUNet3D(
        features=cfg.FEATURES, n_blocks=cfg.N_BLOCKS,
        use_attention=cfg.USE_ATTENTION, use_hybrid_conv=cfg.USE_HYBRID_CONV,
        use_surface_refinement=cfg.USE_SURFACE_REFINEMENT,
        use_deep_supervision=cfg.USE_DEEP_SUPERVISION,
    ).to(cfg.DEVICE)

    print(f"Model: {count_params(model)/1e6:.1f}M params")

    # Loss & Optimizer
    criterion = CombinedLossV11(dice_w=cfg.DICE_WEIGHT, bce_w=cfg.BCE_WEIGHT,
                                 cldice_w=cfg.CLDICE_WEIGHT, surface_w=cfg.SURFACE_WEIGHT,
                                 topo_w=cfg.TOPO_WEIGHT)
    gpu_augment = GPUAugmentation3D(cfg)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LEARNING_RATE, weight_decay=cfg.WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=cfg.EPOCHS_PER_FOLD - cfg.WARMUP_EPOCHS, eta_min=cfg.ETA_MIN)

    # Checkpoint directory
    fold_dir = cfg.CHECKPOINT_DIR / f"fold_{fold}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    # Resume from checkpoint if provided
    start_epoch = 0
    best_dice = 0
    history = []
    
    if resume_from and Path(resume_from).exists():
        start_epoch, best_dice, history = load_checkpoint(
            resume_from, model, optimizer, scheduler, cfg.DEVICE)
    elif cfg.LOAD_CHECKPOINT:
        # Try auto-find latest checkpoint for this fold
        # Search in both save dir (working) and load dir (input dataset)
        latest = get_latest_checkpoint(cfg.CHECKPOINT_DIR, fold, load_dir=cfg.LOAD_CHECKPOINT)
        if latest:
            start_epoch, best_dice, history = load_checkpoint(
                latest, model, optimizer, scheduler, cfg.DEVICE)

    # Compile model (after loading checkpoint)
    if hasattr(torch, 'compile'):
        model = torch.compile(model, mode='reduce-overhead')

    # Precision
    use_bf16 = cfg.USE_BFLOAT16 and torch.cuda.is_bf16_supported()
    dtype = torch.bfloat16 if use_bf16 else torch.float32
    print(f"Precision: {'bfloat16' if use_bf16 else 'float32'}")
    print(f"Starting from epoch {start_epoch + 1}")
    print(f"Checkpoint save strategy:")
    print(f"  - last_checkpoint.pth: Every epoch")
    print(f"  - best_model.pth: When validation improves")
    print(f"  - checkpoint_epoch_N.pth: Every {cfg.SAVE_EVERY} epochs")

    for epoch in range(start_epoch, cfg.EPOCHS_PER_FOLD):
        t0 = time.time()
        model.train()

        # Warmup LR
        if epoch < cfg.WARMUP_EPOCHS:
            for pg in optimizer.param_groups:
                pg['lr'] = get_warmup_lr(epoch, cfg.WARMUP_EPOCHS, cfg.LEARNING_RATE)

        total_loss = 0.0
        n = 0

        pbar = tqdm(train_loader, desc=f"F{fold} E{epoch+1}", file=sys.stdout, leave=False)
        for batch in pbar:
            images = batch['image'].to(cfg.DEVICE, dtype=dtype)
            labels = batch['label'].to(cfg.DEVICE, dtype=dtype)
            ignore = batch['ignore_mask'].to(cfg.DEVICE, dtype=dtype)

            # GPU augmentation (no CPU bottleneck)
            images, labels, ignore = gpu_augment(images, labels, ignore)

            optimizer.zero_grad(set_to_none=True)

            with autocast(device_type='cuda', dtype=dtype):
                out = model(images)
                losses = criterion(out, labels, ignore, epoch)

            losses['total'].backward()
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRADIENT_CLIP)
            optimizer.step()

            total_loss += losses['total'].detach()
            n += 1
            # Only log every 5 batches to avoid CUDA sync
            if n % 5 == 0:
                pbar.set_postfix(loss=f"{losses['total'].item():.3f}", gnorm=f"{grad_norm.item():.2f}")

        if epoch >= cfg.WARMUP_EPOCHS:
            scheduler.step()

        train_loss = total_loss.item() / n
        dt = time.time() - t0

        # Validation
        val_dice = 0
        if (epoch + 1) % cfg.VAL_EVERY == 0:
            val_dice = validate_fast(model, val_loader, cfg.DEVICE, use_bf16)

            if val_dice > best_dice:
                best_dice = val_dice
                save_checkpoint(fold_dir / 'best_model.pth', model, optimizer, scheduler,
                               epoch, best_dice, history, cfg)
                print(f"  >>> New best Dice: {val_dice:.4f}")

        # Log
        lr = optimizer.param_groups[0]['lr']
        log = f"F{fold} E{epoch+1}/{cfg.EPOCHS_PER_FOLD} | {dt:.1f}s | Loss: {train_loss:.4f} | LR: {lr:.1e}"
        if val_dice > 0:
            log += f" | Dice: {val_dice:.4f}"
        print(log)

        history.append({
            'epoch': epoch, 'train_loss': train_loss,
            'lr': lr, 'val_dice': val_dice,
        })

        # =====================================================================
        # ALWAYS save last_checkpoint.pth (for exact resume from any epoch)
        # =====================================================================
        save_checkpoint(fold_dir / 'last_checkpoint.pth', model, optimizer, scheduler,
                       epoch, best_dice, history, cfg)

        # Periodic checkpoint (backup)
        if (epoch + 1) % cfg.SAVE_EVERY == 0:
            save_checkpoint(fold_dir / f'checkpoint_epoch_{epoch+1}.pth',
                           model, optimizer, scheduler, epoch, best_dice, history, cfg)

    print(f"\nFold {fold} complete. Best Dice: {best_dice:.4f}")

    # Save final checkpoint
    save_checkpoint(fold_dir / 'final_model.pth', model, optimizer, scheduler,
                   cfg.EPOCHS_PER_FOLD - 1, best_dice, history, cfg)

    # Save history
    with open(fold_dir / 'history.json', 'w') as f:
        json.dump(history, f, indent=2)

    # Cleanup
    del model, optimizer, scheduler, train_loader, val_loader
    gc.collect()
    torch.cuda.empty_cache()

    return {'fold': fold, 'best_dice': best_dice, 'history': history}

print("Training loop ready (FIXED)")
print("Checkpoint saves:")
print("  - last_checkpoint.pth: EVERY epoch (exact resume)")
print("  - best_model.pth: When validation improves")
print("  - checkpoint_epoch_N.pth: Every SAVE_EVERY epochs")

Training loop ready (FIXED)
Checkpoint saves:
  - last_checkpoint.pth: EVERY epoch (exact resume)
  - best_model.pth: When validation improves
  - checkpoint_epoch_N.pth: Every SAVE_EVERY epochs


In [11]:
# =============================================================================
# CELL 10: RUN TRAINING
# =============================================================================

def run_3fold_cv(resume=False):
    """
    Run 3-fold CV.
    
    Args:
        resume: If True, auto-resume from latest checkpoint for each fold
                Priority: last_checkpoint.pth > checkpoint_epoch_*.pth > best_model.pth
    """
    if not FOLD_SPLITS:
        print("No fold splits available")
        return
    
    results = []
    
    for fold, (train_ids, val_ids) in enumerate(FOLD_SPLITS):
        gc.collect()
        torch.cuda.empty_cache()
        
        resume_path = None
        if resume:
            # Search in both save dir and load dir
            resume_path = get_latest_checkpoint(
                cfg.CHECKPOINT_DIR, 
                fold, 
                load_dir=cfg.LOAD_CHECKPOINT
            )
        
        result = train_fold_v11(fold, train_ids, val_ids, resume_from=resume_path)
        results.append(result)
    
    print("\n" + "="*70)
    print("3-FOLD CV COMPLETE")
    print("="*70)
    for r in results:
        print(f"  Fold {r['fold']}: Dice={r['best_dice']:.4f}")
    print(f"  Mean Dice: {np.mean([r['best_dice'] for r in results]):.4f}")
    print("="*70)
    
    return results


# =============================================================================
# USAGE:
# =============================================================================
# Fresh start (no checkpoints):
#   cv_results = run_3fold_cv(resume=False)
#
# Resume from checkpoints (priority: last > periodic > best):
#   cv_results = run_3fold_cv(resume=True)
#
# Example: If stopped at epoch 47 with best at 45:
#   - last_checkpoint.pth contains epoch 46 state
#   - Will resume from epoch 47 (not 45 or 40)
# =============================================================================

cv_results = run_3fold_cv(resume=True)

Found last_checkpoint.pth in /kaggle/input/models/asharamkanderiwal/v11-v2/pytorch/default/5/checkpoints_v11/fold_0
FOLD 0 TRAINING
Train: 524 | Val: 262
Preloading 524 volumes with robust Z-score normalization...


Loading:   0%|          | 0/524 [00:00<?, ?it/s]

Cached: 524 volumes (83.1 GB)
Dataset: 6288 samples (train)
Preloading 262 volumes with robust Z-score normalization...


Loading:   0%|          | 0/262 [00:00<?, ?it/s]

Cached: 786 volumes (125.1 GB)
Dataset: 1048 samples (val)


...

F0 E631/800 | 959.5s | Loss: 0.5192 | LR: 3.3e-05


Checkpoint saved: /kaggle/working/checkpoints_v11/fold_0/last_checkpoint.pth


F0 E632:   0%|          | 0/1572 [00:00<?, ?it/s]

F0 E632/800 | 958.2s | Loss: 0.5199 | LR: 3.3e-05
Checkpoint saved: /kaggle/working/checkpoints_v11/fold_0/last_checkpoint.pth


F0 E633:   0%|          | 0/1572 [00:00<?, ?it/s]